# 02 — Data Preparation

**Proyek:** SuperStore Business Intelligence (Tableau Public)
**Fase CRISP-DM:** 3. Data Preparation
**Input:** data/raw/SuperStore.xlsx
**Output:** data/clean/superstore_clean.csv dan docs/cleaning_log.md

Notebook ini mengerjakan daftar tindakan dari tabel kualitas data di notebook 01: membersihkan data, membuat fitur baru, memvalidasi hasilnya, lalu mengexport file bersih untuk PostgreSQL, Python (notebook 03), dan Tableau.

> **Prinsip:** file mentah di `data/raw/` **tidak pernah diubah**. Semua perubahan terjadi di salinan di memori, dan setiap perubahan dicatat di cleaning log.

## Daftar Isi
1. Rencana Pembersihan
2. Setup & Load Data
3. Cleaning
4. Feature Engineering
5. Validasi
6. Export & Cleaning Log
7. Ringkasan dan Langkah Berikutnya

## 1. Rencana Pembersihan

Hasil pemeriksaan di notebook 01 diterjemahkan menjadi keputusan berikut: 

| Masalah (notebook 01) | Keputusan | Alasan |
|---|---|---|
| 1 baris duplikat penuh | **Hapus** | Baris kembar yang identik pada semua kolom hampir pasti kesalahan input. |
| 7 baris `Order_ID` + `Product_ID` berulang dengan nilai berbeda | **Pertahankan**, beri `order_line_id` unik | Akan dibuktikan di 3.2: harga per unit sama, hanya jumlahnya yang beda, jadi ini baris pesanan yang sah. |
| 449 kode pos 4 digit | **Tambahkan nol di depan** sampai 5 digit | Nol di depan hilang karena kode pos tersimpan sebagai angka. |
| 17 `Product_Name` punya >1 `Product_ID` | **Pakai `product_id` sebagai kunci produk** | `Product_ID` konsisten: tiap ID hanya punya satu nama. |
| Kolom `Country/Region` hanya 1 nilai | **Buang** | Tidak membawa informasi. |
| Nama kolom tidak seragam (`Sub-Category`, `Country/Region`) | **Ubah ke snake_case** | Memudahkan SQL dan Python. |

## 2. Setup & Load Data

### 2.1 Import library dan tentukan lokasi folder

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

# Cari folder root project secara otomatis: notebook dijalankan dari folder notebook
# (alamat "..") atau dari root project (alamat ".")
ROOT = next((p for p in [Path(".."), Path(".")] if (p / "data/raw/SuperStore.xlsx").exists()), None)
if ROOT is None:
    raise FileNotFoundError("SuperStore.xlsx tidak ditemukan. Cek lokasi file di data/raw/")

RAW_PATH = ROOT / "data/raw/SuperStore.xlsx"
CLEAN_DIR = ROOT / "data/clean"
DOCS_DIR = ROOT / "docs"

# Buat folder output jika belum ada
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

print("Folder root project:", ROOT.resolve())


Folder root project: C:\Github\superstore-analysis


> **Cara membaca hasil:** harus tercetak alamat folder project kamu. Jika muncul `FileNotFoundError`, cek lagi lokasi `SuperStore.xlsx`.

### 2.2 Baca data mentah dan catat angka acuan
Angka acuan dicatat **sebelum** cleaning, dipakai di Bagian 5 untuk membuktikan tidak ada data yang hilang tanpa sengaja.

In [2]:
# Postal_Code dibaca sebagai teks agar tidak diperlukan sebagai angka
raw = pd.read_excel(RAW_PATH, dtype={"Postal_Code": str})

In [3]:
# Angka acuan sebelum cleaning
raw_metrics = {
    "rows": len(raw),
    "order": raw["Order_ID"].nunique(),
    "customers": raw["Customer_ID"].nunique(),
    "sales": raw["Sales"].sum(),
    "profit": raw["Profit"].sum()
}
print(raw_metrics)

# Semua perubahan dikerjakan pada salinan; raw tetap utuh
df = raw.copy()

# Daftar catatan perubahan, diisi setelah tiap langkah cleaning
log = [("Data mentah dibaca", len(df), len(df), "SuperStore.xlsx, 20 kolom")]

{'rows': 9994, 'order': 5009, 'customers': 793, 'sales': np.float64(2297200.8603), 'profit': np.float64(286397.0217)}


> **Cara membaca hasil:** `rows` harus 9994, `orders` 5009, `customers` 793, dengan `sales` ≈ 2.297.201 dan `profit` ≈ 286.397. Ini sama dengan angka di notebook 01.

## 3. Cleaning

### 3.1 Hapus duplikat penuh

In [4]:
before = len(df)

# Simpan dulu baris yang akan dibuang, supaya nilainya bisa dipakai di validasi
dup_rows = df[df.duplicated()]
print("Baris yang dibuang:")
display(dup_rows[["Order_ID", "Product_ID", "Sales", "Quantity", "Profit"]])

# Hapus; keep="first" mempertahankan kemunculan pertama
df = df.drop_duplicates(keep="first")

print(f"Baris: {before:,} -> {len(df):,}")
log.append(("Hapus duplikat penuh", before, len(df), "Baris identik pada semua kolom (order US-2017-150119)"))

Baris yang dibuang:


,Order_ID,Product_ID,Sales,Quantity,Profit
3406,US-2017-150119,FUR-CH-10002965,281.37,2,-12.06


Baris: 9,994 -> 9,993


> **Cara membaca hasil:** 1 baris dibuang, sehingga data menjadi 9.993 baris. Karena `drop_duplicates` mempertahankan kemunculan pertama, tidak ada informasi yang hilang.

### 3.2 Tinjau baris `Order_ID` + `Product_ID` yang berulang
Di notebook 01 ada 7 pasangan yang berulang dengan nilai berbeda. Sebelum memutuskan, kita cek apakah itu **baris yang sah** dengan membandingkan **harga per unit**. Jika harga per unit sama, baris itu hanya produk yang sama dicatat dalam beberapa baris dengan jumlah berbeda.

In [5]:
key = ["Order_ID", "Product_ID"]
rep = df[df.duplicated(key, keep=False)].sort_values(key).copy()

# Harga per unit sebelum diskon tidak tersedia, tetapi Sales/Quantity cukup untuk dibandingkan
rep["Unit_Sales"] = rep["Sales"] / rep["Quantity"]

# Untuk tiap pasangan: berapa nilai Unit_Sales yang berbeda? (1 = harga per unit sama)
cek = rep.groupby(key).agg(
    Jumlah_Baris=("Sales", "size"),
    Quantity=("Quantity", lambda s: list(s)),
    Nilai_Unit_Sales_Berbeda=("Unit_Sales", lambda s: s.round(2).nunique()),
    Diskon_Berbeda=("Discount", "nunique"),
)
cek

,,Jumlah_Baris,Quantity,Nilai_Unit_Sales_Berbeda,Diskon_Berbeda
Order_ID,Product_ID,,,,
CA-2018-103135,OFF-BI-10000069,2,"[9, 6]",1,1
CA-2019-129714,OFF-PA-10001970,2,"[2, 4]",1,1
CA-2019-137043,FUR-FU-10003664,2,"[6, 3]",1,1
CA-2019-140571,OFF-PA-10001954,2,"[14, 2]",1,1
CA-2020-118017,TEC-AC-10002006,2,"[6, 8]",1,1
CA-2020-152912,OFF-ST-10003208,2,"[9, 3]",1,1
US-2019-123750,TEC-AC-10004659,2,"[7, 5]",1,1


> **Cara membaca hasil:** pada ketujuh pasangan, `Nilai_Unit_Sales_Berbeda` bernilai 1 dan `Diskon_Berbeda` bernilai 1. Artinya produk, harga per unit, dan diskonnya sama; yang berbeda hanya jumlahnya (misalnya 9 dan 6 unit). Ini baris pesanan yang sah, jadi **dipertahankan**. Karena `Order_ID` + `Product_ID` tidak unik, kita buat kunci baris yang unik di langkah 3.7.

In [6]:
log.append(("Tinjau Order_ID + Product_ID berulang", len(df), len(df),
            "7 pasangan dipertahankan: harga per unit dan diskon sama, hanya Quantity berbeda"))

### 3.3 Perbaiki `Postal_Code` 

In [7]:
short_before = int((df["Postal_Code"].str.len() < 5).sum())

# zfill(5) menambahkan nol di depan sampai panjangnya 5 karakter
df["Postal_Code"] = df["Postal_Code"].str.zfill(5)

short_after = int((df["Postal_Code"].str.len() < 5).sum())
print(f"Kode pos < 5 digit: {short_before} -> {short_after}")

# Contoh hasil perbaikan
df.loc[df["Postal_Code"].str.startswith("0"), ["State", "City", "Postal_Code"]].drop_duplicates().head()

Kode pos < 5 digit: 449 -> 0


,State,City,Postal_Code
185,Connecticut,Fairfield,06824
197,New Jersey,Westfield,07090
267,New Jersey,Morristown,07960
298,New Jersey,Belleville,07109
306,New Jersey,Lakewood,08701


In [8]:
log.append(("Perbaiki Postal_Code", len(df), len(df), f"{short_before} kode pos diberi nol di depan (zfill 5)"))

> **Cara membaca hasil:** `Kode pos < 5 digit: 449 -> 0`. Pada contoh terlihat kode pos seperti `06824` (Fairfield, Connecticut) dan `07090` (Westfield, New Jersey) kembali utuh. Kolom ini tetap bertipe **teks** supaya nol di depannya tidak hilang lagi.

### 3.4 Rapikan nama kolom

In [9]:
# Pemetaan eksplisit agar jelas nama lama dan baru
rename_map = {
    "Order_ID": "order_id", "Customer_ID": "customer_id", "Customer_Name": "customer_name",
    "Segment": "segment", "Product_ID": "product_id", "Product_Name": "product_name",
    "Category": "category", "Sub-Category": "sub_category",
    "Order_Date": "order_date", "Ship_Date": "ship_date", "Ship_Mode": "ship_mode",
    "Country/Region": "country", "Region": "region", "State": "state", "City": "city",
    "Postal_Code": "postal_code",
    "Sales": "sales", "Quantity": "quantity", "Discount": "discount", "Profit": "profit",
}
df = df.rename(columns=rename_map)
log.append(("Rapikan nama kolom", len(df), len(df), "Semua kolom menjadi snake_case huruf kecil"))
df.columns.tolist()

['order_id',
 'customer_id',
 'postal_code',
 'product_id',
 'sales',
 'quantity',
 'discount',
 'profit',
 'category',
 'sub_category',
 'product_name',
 'order_date',
 'ship_date',
 'ship_mode',
 'customer_name',
 'segment',
 'country',
 'city',
 'state',
 'region']

> **Cara membaca hasil:** semua nama kolom kini huruf kecil dengan garis bawah (`sub_category`, `country`). Tidak ada lagi tanda `-` atau `/` yang merepotkan di SQL.

### 3.5 Buang kolom `country`

In [10]:
print("Nilai unik country:", df["country"].unique().tolist())

df = df.drop(columns="country")
log.append(("Buang kolom country", len(df), len(df), "Hanya satu nilai (United States)"))
print("Jumlah kolom sekarang:", df.shape[1])

Nilai unik country: ['United States']
Jumlah kolom sekarang: 19


> **Cara membaca hasil:** kolom `country` hanya berisi `United States`, jadi dibuang. Jumlah kolom turun dari 20 menjadi 19.

### 3.6 Pastikan `product_id` layak jadi kunci produk
Di notebook 01, 17 nama produk punya lebih dari satu ID. Kita cek bahwa **tiap `product_id`** hanya punya satu nama, satu kategori, dan satu sub-kategori. Itu syarat agar bisa jadi kunci tabel `dim_product` di PostgreSQL.

In [11]:
attrs = df.groupby("product_id").agg(
    n_name=("product_name", "nunique"),
    n_category=("category", "nunique"),
    n_sub_category=("sub_category", "nunique"),
)
print("product_id dengan atribut ganda:")
print((attrs > 1).sum().to_string())

product_id dengan atribut ganda:
n_name            0
n_category        0
n_sub_category    0


In [12]:
log.append(("Validasi product_id sebagai kunci", len(df), len(df),
            "Tiap product_id punya 1 nama, 1 kategori, 1 sub-kategori"))

> **Cara membaca hasil:** semuanya 0, berarti `product_id` aman dijadikan kunci. Nama produk yang sama dengan ID berbeda dibiarkan, karena itu memang produk yang berbeda (misalnya ukuran atau warna) dan tidak ada alasan untuk menggabungkannya.

### 3.7 Buat kunci baris unik `order_line_id`
Sebagai kunci tabel fakta, kita butuh identitas unik per baris. Kita beri nomor urut di dalam tiap order, sesuai urutan di file asli.

In [13]:
# cumcount() menghitung urutan baris di dalam tiap order, mulai dari 0
line_no = df.groupby("order_id").cumcount() + 1
df.insert(0, "order_line_id", df["order_id"] + "-" + line_no.astype(str).str.zfill(2))

print("order_line_id unik:", df["order_line_id"].is_unique)
df[["order_line_id", "order_id", "product_id"]].head()

order_line_id unik: True


,order_line_id,order_id,product_id
0,CA-2019-152156-01,CA-2019-152156,FUR-BO-10001798
1,CA-2019-152156-02,CA-2019-152156,FUR-CH-10000454
2,CA-2019-138688-01,CA-2019-138688,OFF-LA-10000240
3,US-2018-108966-01,US-2018-108966,FUR-TA-10000577
4,US-2018-108966-02,US-2018-108966,OFF-ST-10000760


In [14]:
log.append(("Buat order_line_id", len(df), len(df), "order_id + nomor urut baris, contoh CA-2019-152156-01"))

> **Cara membaca hasil:** `order_line_id unik: True`. Contoh bentuknya `CA-2019-152156-01`. Kunci ini yang nanti jadi *primary key* tabel `fact_sales`.

## 4. Feature Engineering

Kolom turunan dibuat sekarang supaya perhitungannya **satu kali dan konsisten**, lalu dipakai di SQL, notebook 03, dan Tableau.

### 4.1 Fitur waktu

In [15]:
df["order_year"] = df["order_date"].dt.year
df["order_quarter"] = df["order_date"].dt.quarter
df["order_month"] = df["order_date"].dt.month

# Tanggal awal bulan: berguna untuk sumbu waktu bulanan di Tableau
df["order_month_start"] = df["order_date"].dt.to_period("M").dt.to_timestamp()

df[["order_date", "order_year", "order_quarter", "order_month", "order_month_start"]].head()

,order_date,order_year,order_quarter,order_month,order_month_start
0,2019-11-08,2019,4,11,2019-11-01
1,2019-11-08,2019,4,11,2019-11-01
2,2019-06-12,2019,2,6,2019-06-01
3,2018-10-11,2018,4,10,2018-10-01
4,2018-10-11,2018,4,10,2018-10-01


> **Cara membaca hasil:** satu tanggal pemesanan kini punya tahun, kuartal, bulan, dan tanggal awal bulan. Misalnya `2019-11-08` menjadi tahun 2019, kuartal 4, bulan 11, dan awal bulan `2019-11-01`.

### 4.2 Lama pengiriman

In [16]:
# Buang format lama yang rusak, lalu pasang format yang benar (tanpa spasi)
pd.reset_option("display.float_format")
pd.set_option("display.float_format", "{:,.2f}".format)

In [17]:
df["ship_days"] = (df["ship_date"] - df["order_date"]).dt.days

df["ship_days"].describe().to_frame("ship_days").T

,count,mean,std,min,25%,50%,75%,max
ship_days,"9,993.00",3.96,1.75,0.00,3.00,4.00,5.00,8.00


> **Cara membaca hasil:** `min` 0 dan `max` 8 hari, dengan rata-rata sekitar 4 hari (3,96). Rentangnya sama dengan di notebook 01, karena baris duplikat yang dibuang tidak memengaruhinya.

### 4.3 Profit margin dan penanda transaksi rugi

In [18]:
# Margin per baris = profit / sales (sales selalu > 0, sudah dicek di notebook 01)
df["profit_margin"] = df["profit"] / df["sales"]

# is_loss: 1 jika transaksi rugi, 0 jika tidak (angka 0/1 mudah dijumlahkan di SQL dan Tableau)
df["is_loss"] = (df["profit"] < 0).astype(int)

print(f"Transaksi rugi: {df['is_loss'].sum():,} ({df['is_loss'].mean():.1%})")
df["profit_margin"].describe().to_frame("profit_margin").T

Transaksi rugi: 1,870 (18.7%)


,count,mean,std,min,25%,50%,75%,max
profit_margin,"9,993.00",0.12,0.47,-2.75,0.07,0.27,0.36,0.50


> **Cara membaca hasil:** transaksi rugi tinggal 1.870 baris (18,7%), karena satu baris duplikat yang rugi sudah dibuang. `profit_margin` berkisar dari sekitar −275% sampai +50%. Margin −275% artinya rugi hampir 3 kali nilai penjualan pada baris tersebut, jadi ada transaksi dengan kerugian ekstrem.
>
> **Hati-hati:** `profit_margin` dan `is_loss` dibuat dari `profit`, sehingga **tidak boleh dipakai sebagai fitur** untuk memprediksi kerugian di notebook 03 (itu membocorkan jawabannya).

### 4.4 Kelompok diskon
Diskon dikelompokkan supaya mudah dibandingkan di dashboard. Batasnya dipilih sesuai pola di notebook 01: profit mulai negatif di atas diskon 20%.

In [19]:
bins = [-0.001, 0, 0.20, 0.40, 0.60, 0.80]
labels = ["0%", "1-20%", "21-40%", "41-60%", "61-80%"]

# pd.cut membagi angka ke dalam rentang; batas kanan termasuk (misal 0.20 masuk "1-20%")
df["discount_band"] = pd.cut(df["discount"], bins=bins, labels=labels)

(df.groupby("discount_band", observed=True)
   .agg(baris=("profit", "size"), rata_rata_profit=("profit", "mean"), total_profit=("profit", "sum")))

,baris,rata_rata_profit,total_profit
discount_band,,,
0%,4798,66.90,"320,987.60"
1-20%,3803,26.50,"100,785.47"
21-40%,459,-78.01,"-35,805.41"
41-60%,215,-134.62,"-28,944.19"
61-80%,718,-98.35,"-70,614.40"


> **Cara membaca hasil:** tanpa diskon, rata-rata profit per baris sekitar +66,9 dan diskon 1–20% sekitar +26,5. Mulai kelompok **21–40%**, rata-rata profit menjadi negatif (sekitar −78) dan makin buruk di kelompok berikutnya. Pola ini membuat `discount_band` jadi dimensi penting untuk dashboard.

### 4.5 Fitur pelanggan: pesanan pertama dan cohort
*Cohort* adalah kelompok pelanggan menurut tahun pertama kali mereka membeli.

In [20]:
# transform("min") menaruh tanggal order pertama tiap pelanggan di setiap barisnya
df["first_order_date"] = df.groupby("customer_id")["order_date"].transform("min")
df["cohort_year"] = df["first_order_date"].dt.year

# Satu pelanggan = satu cohort
cohort = df.drop_duplicates("customer_id")["cohort_year"].value_counts().sort_index()
cohort.to_frame("jumlah_pelanggan")

,jumlah_pelanggan
cohort_year,
2017,595
2018,136
2019,51
2020,11


> **Cara membaca hasil:** 595 pelanggan pertama kali membeli di 2017, 136 di 2018, 51 di 2019, dan 11 di 2020 (total 793). Sebagian besar pelanggan sudah ada sejak 2017, jadi pertumbuhan Sales di 2019–2020 lebih banyak berasal dari **pelanggan yang sama yang membeli lebih banyak**, bukan pelanggan baru.

In [21]:
log.append(("Buat fitur waktu", len(df), len(df), "order_year, order_quarter, order_month, order_month_start"))
log.append(("Buat ship_days", len(df), len(df), "ship_date - order_date (hari)"))
log.append(("Buat profit_margin dan is_loss", len(df), len(df), "profit/sales dan penanda rugi 0/1"))
log.append(("Buat discount_band", len(df), len(df), "5 kelompok: 0%, 1-20%, 21-40%, 41-60%, 61-80%"))
log.append(("Buat fitur pelanggan", len(df), len(df), "first_order_date dan cohort_year"))

## 5. Validasi

Sebelum export, kita buktikan bahwa data bersih **konsisten dengan data mentah**: perbedaannya harus persis sama dengan satu baris duplikat yang dibuang.

### 5.1 Rekonsiliasi dengan data mentah

In [22]:
# Nilai yang diharapkan = angka mentah dikurangi baris duplikat yang dibuang
expected = {
    "rows": raw_metrics["rows"] - len(dup_rows),
    "order": raw_metrics["order"],
    "customers": raw_metrics["customers"],
    "sales": raw_metrics["sales"] - dup_rows["Sales"].sum(),
    "profit": raw_metrics["profit"] - dup_rows["Profit"].sum(),
}
actual = {
    "rows": len(df),
    "orders": df["order_id"].nunique(),
    "customers": df["customer_id"].nunique(),
    "sales": df["sales"].sum(),
    "profit": df["profit"].sum(),
}

recon = pd.DataFrame({"mentah": raw_metrics, "diharapkan": expected, "bersih": actual})
recon["status"] = [
    "OK" if np.isclose(recon.loc[k, "diharapkan"], recon.loc[k, "bersih"]) else "BERBEDA"
    for k in recon.index
]
recon

,mentah,diharapkan,bersih,status
rows,"9,994.00","9,993.00","9,993.00",OK
order,"5,009.00","5,009.00",NaN,BERBEDA
customers,793.00,793.00,793.00,OK
sales,"2,297,200.86","2,296,919.49","2,296,919.49",OK
profit,"286,397.02","286,409.08","286,409.08",OK
orders,NaN,NaN,"5,009.00",BERBEDA


> **Cara membaca hasil:** kolom `status` semuanya harus `OK`. Baris berkurang 1 (9.994 menjadi 9.993), sedangkan jumlah order dan pelanggan tidak berubah. Selisih Sales dan Profit sama persis dengan baris duplikat yang dibuang (≈ 281,37 dan ≈ −12,06). Jika ada `BERBEDA`, berarti ada langkah cleaning yang mengubah data tanpa sengaja.

### 5.2 Pemeriksaan kualitas data bersih

In [23]:
checks = {
    "Missing value": int(df.isna().sum().sum()),
    "order_line_id tidak unik": int(not df["order_line_id"].is_unique),
    "Duplikat penuh": int(df.duplicated().sum()),
    "Kode pos bukan 5 digit": int((df["postal_code"].str.len() != 5).sum()),
    "ship_days negatif": int((df["ship_days"] < 0).sum()),
    "sales <= 0": int((df["sales"] <= 0).sum()),
    "discount di luar 0-1": int((~df["discount"].between(0, 1)).sum()),
    "discount_band kosong": int(df["discount_band"].isna().sum()),
}
pd.Series(checks, name="jumlah_masalah").to_frame()

,jumlah_masalah
Missing value,0
order_line_id tidak unik,0
Duplikat penuh,0
Kode pos bukan 5 digit,0
ship_days negatif,0
sales <= 0,0
discount di luar 0-1,0
discount_band kosong,0


> **Cara membaca hasil:** semua nilai harus 0. Jika ada yang tidak nol, jangan export dulu; telusuri langkah cleaning yang bersangkutan.

### 5.3 Susunan kolom akhir

In [24]:
final_cols = [
    "order_line_id", "order_id", "order_date", "ship_date", "ship_mode",
    "order_year", "order_quarter", "order_month", "order_month_start", "ship_days",
    "customer_id", "customer_name", "segment", "first_order_date", "cohort_year",
    "region", "state", "city", "postal_code",
    "product_id", "product_name", "category", "sub_category",
    "sales", "quantity", "discount", "discount_band", "profit", "profit_margin", "is_loss",
]

# Pastikan tidak ada kolom yang terlewat atau berlebih
assert set(final_cols) == set(df.columns), set(final_cols) ^ set(df.columns)

df_clean = df[final_cols].sort_values(["order_date", "order_line_id"]).reset_index(drop=True)
print(f"Data bersih: {df_clean.shape[0]:,} baris x {df_clean.shape[1]} kolom")
df_clean.dtypes.to_frame("tipe")

Data bersih: 9,993 baris x 30 kolom


,tipe
order_line_id,str
order_id,str
order_date,datetime64[us]
ship_date,datetime64[us]
ship_mode,str
order_year,int32
order_quarter,int32
order_month,int32
order_month_start,datetime64[us]
ship_days,int64


> **Cara membaca hasil:** `9,993 baris x 30 kolom` (19 kolom asli ditambah 11 kolom baru). Kolom tanggal bertipe `datetime64`, kolom angka bertipe angka (`int` atau `float`), kolom teks bertipe `str`, dan `discount_band` bertipe kategori.

## 6. Export & Cleaning Log

### 6.1 Export data bersih ke CSV

In [25]:
CLEAN_PATH = CLEAN_DIR / "superstore_clean.csv"

# Tanggal ditulis dengan format ISO (YYYY-MM-DD) agar dibaca benar oleh PostgreSQL dan Tableau
export = df_clean.copy()
for col in ["order_date", "ship_date", "order_month_start", "first_order_date"]:
    export[col] = export[col].dt.strftime("%Y-%m-%d")
export["discount_band"] = export["discount_band"].astype(str)

export.to_csv(CLEAN_PATH, index=False, encoding="utf-8")
print("Tersimpan:", CLEAN_PATH.resolve())
print(f"Ukuran: {CLEAN_PATH.stat().st_size / 1024:,.0f} KB")

Tersimpan: C:\Github\superstore-analysis\data\clean\superstore_clean.csv
Ukuran: 2,807 KB


> **Cara membaca hasil:** muncul alamat file `superstore_clean.csv` dengan ukuran sekitar 2,8 MB.

### 6.2 Uji baca ulang file hasil export
Kita pastikan file yang tersimpan terbaca kembali dengan benar, terutama kode pos yang diawali nol.

In [26]:
# Baca ulang; postal_code harus dibaca sebagai teks agar nol di depannya tidak hilang
check = pd.read_csv(CLEAN_PATH, dtype={"postal_code": str}, parse_dates=["order_date", "ship_date"])

print("Bentuk:", check.shape)
print("Kode pos kurang dari 5 digit:", int((check["postal_code"].str.len() < 5).sum()))
print("Total sales sama dengan data bersih:", np.isclose(check["sales"].sum(), df_clean["sales"].sum()))
check.head(3)

Bentuk: (9993, 30)
Kode pos kurang dari 5 digit: 0
Total sales sama dengan data bersih: True


,order_line_id,order_id,order_date,ship_date,ship_mode,order_year,order_quarter,order_month,order_month_start,ship_days,customer_id,customer_name,segment,first_order_date,cohort_year,region,state,city,postal_code,product_id,product_name,category,sub_category,sales,quantity,discount,discount_band,profit,profit_margin,is_loss
0,CA-2017-103800-01,CA-2017-103800,2017-01-03,2017-01-07,Standard Class,2017,1,1,2017-01-01,4,DP-13000,Darren Powers,Consumer,2017-01-03,2017,Central,Texas,Houston,77095,OFF-PA-10000174,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",Office Supplies,Paper,16.45,2,0.20,1-20%,5.55,0.34,0
1,CA-2017-112326-01,CA-2017-112326,2017-01-04,2017-01-08,Standard Class,2017,1,1,2017-01-01,4,PO-19195,Phillina Ober,Home Office,2017-01-04,2017,Central,Illinois,Naperville,60540,OFF-LA-10003223,Avery 508,Office Supplies,Labels,11.78,3,0.20,1-20%,4.27,0.36,0
2,CA-2017-112326-02,CA-2017-112326,2017-01-04,2017-01-08,Standard Class,2017,1,1,2017-01-01,4,PO-19195,Phillina Ober,Home Office,2017-01-04,2017,Central,Illinois,Naperville,60540,OFF-ST-10002743,SAFCO Boltless Steel Shelving,Office Supplies,Storage,272.74,3,0.20,1-20%,-64.77,-0.24,1


> **Cara membaca hasil:** bentuknya `(9993, 30)`, kode pos kurang dari 5 digit = 0, dan total sales sama (`True`).
>
> **Catatan untuk Tableau:** saat Tableau membaca CSV, kolom `postal_code` bisa terdeteksi sebagai angka. Jika itu terjadi, ubah tipe datanya menjadi **String** di tab Data Source supaya nol di depan tidak hilang. Hal ini diulang di panduan dashboard.

### 6.3 Tulis cleaning log
Cleaning log mencatat setiap perubahan, sehingga proses pembersihan bisa ditelusuri dan diulang.

In [27]:
log_df = pd.DataFrame(log, columns=["Langkah", "Baris sebelum", "Baris sesudah", "Keterangan"])
log_df.insert(0, "No", range(1, len(log_df) + 1))

# Susun tabel Markdown secara manual (tanpa library tambahan)
lines = [
    "# Cleaning Log — SuperStore",
    "",
    "Dihasilkan oleh `notebook/02_superstore_data_preparation.ipynb`.",
    "",
    "| No | Langkah | Baris sebelum | Baris sesudah | Keterangan |",
    "|---|---|---|---|---|",
]
for r in log_df.itertuples(index=False):
    lines.append(f"| {r[0]} | {r[1]} | {r[2]:,} | {r[3]:,} | {r[4]} |")

lines += [
    "",
    "## Hasil akhir",
    f"- Baris: {len(df_clean):,} (mentah: {raw_metrics['rows']:,})",
    f"- Kolom: {df_clean.shape[1]}",
    "- Output: `data/clean/superstore_clean.csv`",
]

(DOCS_DIR / "cleaning_log.md").write_text("\n".join(lines), encoding="utf-8")
log_df

,No,Langkah,Baris sebelum,Baris sesudah,Keterangan
0,1,Data mentah dibaca,9994,9994,"SuperStore.xlsx, 20 kolom"
1,2,Hapus duplikat penuh,9994,9993,Baris identik pada semua kolom (order US-2017-...
2,3,Tinjau Order_ID + Product_ID berulang,9993,9993,7 pasangan dipertahankan: harga per unit dan d...
3,4,Perbaiki Postal_Code,9993,9993,449 kode pos diberi nol di depan (zfill 5)
4,5,Rapikan nama kolom,9993,9993,Semua kolom menjadi snake_case huruf kecil
5,6,Buang kolom country,9993,9993,Hanya satu nilai (United States)
6,7,Validasi product_id sebagai kunci,9993,9993,"Tiap product_id punya 1 nama, 1 kategori, 1 su..."
7,8,Buat order_line_id,9993,9993,"order_id + nomor urut baris, contoh CA-2019-15..."
8,9,Buat fitur waktu,9993,9993,"order_year, order_quarter, order_month, order_..."
9,10,Buat ship_days,9993,9993,ship_date - order_date (hari)


> **Cara membaca hasil:** tabel di atas juga tersimpan sebagai `docs/cleaning_log.md`. File itu bisa ditautkan di README sebagai bukti proses cleaning yang terdokumentasi.

## 7. Ringkasan dan Langkah Berikutnya

### 7.1 Yang sudah dikerjakan
- Menghapus **1 duplikat penuh** (9.994 → 9.993 baris).
- Memutuskan mempertahankan **7 baris** `Order_ID` + `Product_ID` berulang, karena harga per unit dan diskonnya sama, dengan kunci baru `order_line_id`.
- Memperbaiki **449 kode pos** dan membuang kolom `country`.
- Membuat **11 kolom baru**: fitur waktu, `ship_days`, `profit_margin`, `is_loss`, `discount_band`, `first_order_date`, `cohort_year`, dan `order_line_id`.
- Validasi: total Sales dan Profit cocok dengan data mentah setelah dikurangi baris duplikat.

### 7.2 Output
| File | Fungsi |
|---|---|
| `data/clean/superstore_clean.csv` | Sumber data untuk PostgreSQL, notebook 03, dan Tableau |
| `docs/cleaning_log.md` | Dokumentasi proses cleaning untuk README |

### 7.3 Langkah berikutnya
- [ ] **Fase SQL (PostgreSQL + DBeaver):** import `superstore_clean.csv`, buat star schema (`fact_sales`, `dim_customer`, `dim_product`, `dim_date`, `dim_geo`), dan view KPI.
- [ ] **Notebook 03 (modelling):** RFM + KMeans dan klasifikasi transaksi rugi (tanpa memakai `profit`, `profit_margin`, atau `is_loss` sebagai fitur).
- [ ] **Tableau Public:** dashboard dari hasil view KPI.